# Electric Vehicles in `echo`

Electric vehicles (EVs) are currently modelled as stationary objects in `echo`. They form a single connection with a connection point (`Tellegen`) node, which is to say that they only charge or discharge from a single location. Electric vehicles that can change locations will be introduced in future versions of `echo`.   

There are four types of electric vehicle (EV) in `echo`:
1. `EVV0G`
2. `EVV1G`
3. `EVV2G`
4. `EVWithProfile`

`EVV0G`, `EVV1G` and `EVV2G` are similiar objects in that their charging (and discharging) profiles are determined around the `available` attribute; a boolean timeseries array that describes whether the connection to the connection point is inactive (0) or active (1). If the connection is inactive the vehicle may consume energy, which represents the vehicle travelling. This energy consumption is described by the `usage` attribute. 

`EVWithProfile` is primarily intended to be used when real world data is available. It is different from the other three EV objects in that charging is described solely from the `demand` attribute. `demand` is a time series data that contains charging data. Currently `EVWithProfile` can only describe charging, but this will be changed to charging and discharging in a future release of `echo`.  

## `EVBase` - The Parent Class of `EVV0G`, `EVV1G` and `EVV2G`

`EVV0G`, `EVV1G` and `EVV2G` classes have a lot in common; the primary difference between the three is the way that charging and discharging is calculated. Let's first have a look at the attributes that they share:

In [ ]:
from echo.models.electrical import EVBase

print(EVBase.__doc__)

### `EVV0G` - Convenience Charging
Convenience charging means there is no demand or generation management taking place. Once the `EVV0G` is available to charge, it will charge at the maximum possible power until either the EV's battery is full, or the EV becomes unavailable to charge. The `tod_charging` can be used to replicate timers within vehicles that allows or forbids charging at certain times, separate to the `available` attribute.    
### `EVV1G` - Demand Managed Charging
Charging of the EV is managed by the optimiser. Providing energy back to the electrical network is not possible, i.e. the flow of energy to the EV $\ge$ 0. 

### `EVV2G` - Demand and Generation Managed Charging
Charging of the EV and discharging of the EV to the electrical network is managed by the optimiser. If `usage_power_limit=None` then the discharging power limit to the grid, as well as through usage on a trip, will be the same value of `discharging_power_limit`. If `usage_power_limit` is set to a `float`, then `usage_power_limit` describes the maximum power output on a trip, while `discharging_power_limit` describes the maximum power output to the electrical network. 

## `EVWithProfile` - Profile Dictated Charging

In [ ]:
from echo.models.electrical import EVWithProfile

print(EVWithProfile.__doc__)

## Examples

Let's start by importing everything we're going to need

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pyomo.util.infeasible import log_infeasible_constraints

from echo.configuration import Units
from echo.models.agnostic import FlexPort, TellegenNode
from echo.models.base import OptimisationGraph
from echo.models.electrical import EVV0G, EVV1G, EVV2G, EVWithProfile
from echo.models.prebuilt import FlexElectricalNode
from echo.models.scenario import ScenarioSettings, engine_settings_from_environment
from echo.objectives.base import ObjectiveSet
from echo.objectives.tariff import ImportTariff, ThroughputCost
from echo.optimiser import optimise

## Simple System with Each EV Object

First we need to define some parameters to set the scene for out simulation. In this simulation we will have 48 data points that are spaced 30 minutes apart, i.e. we are simulating a 24 hour period.

In [ ]:
# Number of time periods to run the optimisation for
time_periods = 48

# The duration in minutes between each data point in time series data
interval_duration = 30

Next we begin to build our system. It will be a simple system composing of a `Grid` object (an infinite source and sink of energy), a connection point (a Tellegen node) and one of each of the EV's connected to the connection point. 

In [ ]:
# Create an infinite grid node with one downstream port
grid = FlexElectricalNode(node_name="grid", port_name="connection_point")

# Create a connection point (zero sum) node with ports for our three EVs
connection_point = TellegenNode(node_name="connection_point")
connection_point.add_ports_from_list(
    ["grid", "ev_v0g", "ev_v1g", "ev_v2g", "ev_with_profile"],
    FlexPort,
    units=Units.KW,
)

# Create availability and usage data for a V0G vehicle
ev_v0g_available = [1] * 36 + [0] * 12
ev_v0g_usage = [0] * 36 + [1.5] * 12

# Create V0G vehicle
ev_v0g = EVV0G(
    node_name="ev_v0g",
    available=ev_v0g_available,
    usage=ev_v0g_usage,
    connection_port_name="cp",
    max_capacity=40,
    depth_of_discharge_limit=0,
    charging_power_limit=10,
    discharging_power_limit=-10,
    charging_efficiency=1,
    discharging_efficiency=1,
    initial_state_of_charge=20,
    soc_conserv=None,
    soc_conserv_cost=0.0,
    interval_duration=interval_duration,
    tod_charging=None,
    trip_slack=False,
)

# Create availability and usage data for a V1G vehicle
ev_v1g_available = np.array([1] * 36 + [0] * 12)
ev_v1g_usage = np.array([0.0] * 36 + [1.5] * 12)

# Create V1G vehicle
ev_v1g = EVV1G(
    node_name="ev_v1g",
    available=ev_v1g_available,
    usage=ev_v1g_usage,
    connection_port_name="cp",
    max_capacity=40,
    depth_of_discharge_limit=0,
    charging_power_limit=1,
    discharging_power_limit=-10,
    charging_efficiency=1,
    discharging_efficiency=1,
    initial_state_of_charge=0,
    soc_conserv=None,
    soc_conserv_cost=0.0,
    interval_duration=interval_duration,
    tod_charging=False,
    trip_slack=False,
)

# Create availability and usage data for a V2G vehicle
ev_v2g_available = np.array([1] * 24 + [0] * 24)
ev_v2g_usage = np.array([0.0] * 24 + [1.5] * 24)

# Create a V2G vehicle
ev_v2g = EVV2G(
    node_name="ev_v2g",
    available=ev_v2g_available,
    usage=ev_v2g_usage,
    connection_port_name="cp",
    max_capacity=40,
    depth_of_discharge_limit=0,
    charging_power_limit=10,
    discharging_power_limit=-5,
    charging_efficiency=1,
    discharging_efficiency=1,
    initial_state_of_charge=20,
    soc_conserv=None,
    soc_conserv_cost=1.0,
    interval_duration=interval_duration,
    tod_charging=False,
    trip_slack=False,
    set_stateful_attrs_at_init=True,
    usage_power_limit=-10,
)

# Create demand data for a EVWithProfile vechicle.
ev_with_profile_demand = [5] * 24 + [0] * 12 + [9] * 12

# Create a EVWithProfile
ev_with_profile = EVWithProfile(
    node_name="ev_with_profile",
    port_name="cp",
    charging_power_limit=10,
    set_stateful_attrs_at_init=True,
    demand=ev_with_profile_demand,
)

Each of the nodes that we have just created will be part of the `OptimisationGraph`. So let's create an `OptimisationGraph` and add he nodes to it. 

In [ ]:
# Create the optmisation graph
system = OptimisationGraph()

# Add nodes to the optimisation graph
system.add_node_obj(
    [
        grid,
        connection_point,
        ev_v0g,
        ev_v1g,
        ev_v2g,
        ev_with_profile,
    ]
)

Now need to connect each of the nodes such that we end up with a graph that looks like:

```
grid
 |
connection_point
 |
 __________________________
 |       |       |        |
 ev_v0g  ev_v1g  ev_v2g  ev_with_profile
```

In [ ]:
# Create edge objects and add to graph
system.connect_ports_and_create_edge(grid.ports["connection_point"], connection_point.ports["grid"])
system.connect_ports_and_create_edge(connection_point.ports["ev_v0g"], ev_v0g.ports["cp"])
system.connect_ports_and_create_edge(connection_point.ports["ev_v1g"], ev_v1g.ports["cp"])
system.connect_ports_and_create_edge(connection_point.ports["ev_v2g"], ev_v2g.ports["cp"])
system.connect_ports_and_create_edge(connection_point.ports["ev_with_profile"], ev_with_profile.ports["cp"])

Let's check that the network looks the same as how we envisioned it:

In [ ]:
network_figure = plt.figure()
network_axes = network_figure.add_subplot()
style = {
    "edgecolors": "#FFFFFF",  # node border color
    "linewidths": 3,  # node border width
    "edge_color": "#CCCCCC",  # edge color of edges
    "width": 2,  # edge width
    "horizontalalignment": "left",  # label horizontal position
    "verticalalignment": "top",  # label vertical position
    "font_color": "#555555",  # label text color
}
system.draw_on_axes(axes=network_axes, with_labels=True, **style)
plt.show()

It's a starfish shape instead of a tree shape, but topologically they're the same.

Let's now define the objectives we wish to use in this optimisation. The only controllable devices we have are the `EVV1G` and `EVV2G` nodes. So let's try to minimise the import cost to both of these EVs. 

We'll define a sawtooth-like import tariff profile to make the results stand out.

We'll also define a `ThroughputCost`, which is effectively a cost (in the optimisation sense of the word) of switch between not charging and charging. This has the effect of disincentivising the rapid change between charging and not-charging solutions that the optimiser may find.

In [ ]:
# Define the sawtooth-like import tariff
import_tariff = [10, 5] * 24

# Define the minimise import cost objective for the ev_v1g
minimise_import_cost_ev_v1g = ImportTariff(
    component=ev_v1g.ports["cp"],
    tariff_array=import_tariff,
)

# Define the minimise import cost objective for the ev_v2g
minimise_import_cost_ev_v2g = ImportTariff(
    component=ev_v2g.ports["cp"],
    tariff_array=import_tariff,
)

# Define a minimise throughput cost objective on the ev_v1g's battery.
minimise_throughput_cost_ev_v1g = ThroughputCost(
    component=ev_v1g.ports["cp"],
    rate=0.1,
)

# Define a minimise throughput cost objective on the ev_v2g's battery.
minimise_throughput_cost_ev_v2g = ThroughputCost(
    component=ev_v2g.ports["cp"],
    rate=0.1,
)

We combine all of these objectives into the `ObjectiveSet` object. 

In [ ]:
# Define the objective set
objective_set = ObjectiveSet(
    objective_list=[
        minimise_import_cost_ev_v1g,
        minimise_import_cost_ev_v2g,
        minimise_throughput_cost_ev_v1g,
        minimise_throughput_cost_ev_v2g,
    ]
)

The last thing we need to do is to run the optimisation. 

Here we are using `engine_settings_from_environment()` to tell pyomo which optimiser to use, which defaults to `cplex`. To change this you will need to run:
```
export OPTIMISER_ENGINE="cbc"
```
in the terminal/environment that you launched this notebook from.

In [ ]:
# Invoke the optimiser and optimise
optimise_results = optimise(
    scenario_settings=ScenarioSettings(
        interval_duration=interval_duration,
        number_of_intervals=time_periods,
        number_of_expansion_intervals=1,
        discount_rate=0,
    ),
    engine_settings=engine_settings_from_environment(),
    graph=system,
    objective_set=None,
)

You will see warnings like the above being raised when using EVs in `echo`: Ports {port_names} are defined on nodes but are not part of an edge. You don't need to worry about these; they are due to the internal construction of the EV objects in `echo` which will be updated in a future `echo` version.

Let's have a look at some of the results of the optimisation by plotting them.

In [ ]:
# Plot ev_v0g usage and state of charge
plt.plot(ev_v0g_usage)
plt.plot(optimise_results.values(ev_v0g.ports["vehicle"].soc_value, 0))
plt.legend(["Usage", "EV State of Charge"])
plt.xlim([0, 47])
plt.show()

# Plot ev_v1g usage, state of charge and import tariff
plt.plot(ev_v1g_usage)
plt.plot(optimise_results.values(ev_v1g.ports["vehicle"].soc_value, 0))
plt.plot(import_tariff)
plt.legend(["Usage", "EV soc", "Import Tariff"])
plt.xlim([0, 47])
plt.show()

# Plot ev_v2g usage, state of charge and import tariff
plt.plot(ev_v2g_usage)
plt.plot(optimise_results.values(ev_v2g.ports["vehicle"].soc_value, 0))
plt.plot(import_tariff)
plt.legend(["Usage", "EV soc", "Import Tariff"])
plt.xlim([0, 47])
plt.show()

# Plot ev_with_profile input demand, output demand (which should be the same as input demand) and import tariff. There is no state
# of charge value for ev_with_profile
plt.plot(ev_with_profile_demand)
plt.plot(optimise_results.node_values(ev_with_profile, 0)["cp"])
plt.plot(import_tariff)
plt.legend(["Input Demand", "Output Demand", "Import Tariff"])
plt.xlim([0, 47])
plt.show()